# Voice × CRM: воспроизводимая проверка истории клиента и простого LLM-подхода

Ноутбук предназначен для запуска в Sigma.

Цель: пройти путь от исходных файлов к проверяемой аналитике без “магии” готового Excel:

1. собрать минимально нужный Voice-слой из диалогов, дат и файла `Для метчинга.xlsx`;
2. прочитать исходные CRM-файлы Ани (`part-*.csv`);
3. показать инвентаризацию полей и заполненность;
4. сматчить Voice с CRM по `Id задачи` и `Id ПрПр`;
5. построить историю CRM до звонка;
6. сравнить простые режимы подачи истории в LLM;
7. на маленьком сэмпле протестировать `GigaChat-3-Ultra`, `GigaChat-3-Ultra-Preview`, `GigaChat-3-Ultra-B200`.

Важный принцип: сначала проверяем простой baseline, потом усложняем только если качество недостаточно.

## 0. Как пользоваться

1. Положите рядом с ноутбуком:
   - `Датасет диалогов КМ ЗП 01.01.26-18.04.26.xlsx`
   - `дни.xlsx`
   - `Для метчинга.xlsx` с колонками `ucid`, `Id задачи`, `Id ПрПр`
   - CRM-файлы `part-*.csv`
   - сертификаты GigaChat в `certs/mp/`
2. Запустите ячейки до блока LLM. Это не отправляет данные во внешние сервисы.
3. Проверьте инвентаризацию, матчинг и сэмпл.
4. Для маленького LLM-прогона поставьте `RUN_GIGACHAT = True`.

По умолчанию API-прогон выключен.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 260)
warnings.simplefilter("ignore", category=pd.errors.ParserWarning)

## 1. Настройки

In [ ]:
BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / "sigma_crm_results"
OUT_DIR.mkdir(exist_ok=True)

# Входные файлы. Если автоимя не сработает, укажите путь явно.
DIALOGUES_XLSX = BASE_DIR / "Датасет диалогов КМ ЗП 01.01.26-18.04.26.xlsx"
DAYS_XLSX = BASE_DIR / "дни.xlsx"
MATCH_XLSX = BASE_DIR / "Для метчинга.xlsx"
CRM_GLOB = "part-*.csv"

# Период анализа.
CALL_START = pd.Timestamp("2026-02-01")
CALL_END_EXCLUSIVE = pd.Timestamp("2026-03-01")
CRM_HISTORY_START = pd.Timestamp("2025-12-01")
CRM_HISTORY_END_EXCLUSIVE = pd.Timestamp("2026-03-01")

# Режимы подачи истории модели.
HISTORY_MODES = [
    "salary_only",       # только события с product_group_code ~ SALARY
    "salary_or_text",    # SALARY + события, где текст похож на ЗП-контекст
    "all_recent",        # вся свежая история, но ЗП-события сортируются выше
]
MAX_EVENTS_PER_CALL = 10

# Маленький LLM-сэмпл.
SAMPLE_SIZE = 45
RANDOM_SEED = 42
TEST_HISTORY_MODES = ["salary_only", "all_recent"]

# GigaChat. Прогон выключен, пока не проверены входы и сэмпл.
RUN_GIGACHAT = False
GIGACHAT_MODELS = [
    "GigaChat-3-Ultra",
    "GigaChat-3-Ultra-Preview",
    "GigaChat-3-Ultra-B200",
]
GIGACHAT_BASE_URL = "https://gigachat-ift.sberdevices.delta.sbrf.ru/v1"
CERT_DIR = BASE_DIR / "certs" / "mp"
CERT_FILE = CERT_DIR / "gigachat_ca.pem"
KEY_FILE = CERT_DIR / "gigachat.key"
N_WORKERS = 4  # если новые сертификаты держат 10 потоков, можно поставить 10
LLM_TIMEOUT = 180

print("Рабочая папка:", BASE_DIR)
print("Папка результатов:", OUT_DIR)

## 2. Общие функции

In [ ]:
def clean(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, float) and math.isnan(value):
        return ""
    text = str(value).strip()
    if text.lower() in {"", "nan", "none", "null", "[null]", "[]"}:
        return ""
    return text


def norm_id(value: Any) -> str:
    text = clean(value)
    if not text:
        return ""
    if re.fullmatch(r"\d+\\.0", text):
        text = text[:-2]
    return text.strip().strip("'").strip('"')


def canon_name(name: Any) -> str:
    s = str(name).lower().replace("ё", "е")
    s = s.replace("id", "id").replace("ed", "id")
    return re.sub(r"[\s_()/\\.-]+", "", s)


def find_col(df: pd.DataFrame, exact=(), regex=(), required=False, label="") -> str | None:
    exact_canons = {canon_name(x) for x in exact}
    for col in df.columns:
        if canon_name(col) in exact_canons:
            return col
    for col in df.columns:
        cn = canon_name(col)
        if any(re.search(pattern, cn) for pattern in regex):
            return col
    if required:
        raise KeyError(f"Не найдена обязательная колонка {label or exact or regex}. Доступные колонки: {list(df.columns)}")
    return None


def coalesce_cols(df: pd.DataFrame, cols: list[str | None]) -> pd.Series:
    out = pd.Series(pd.NA, index=df.index, dtype="object")
    for col in cols:
        if col and col in df.columns:
            values = df[col].map(clean)
            out = out.where(out.map(clean).ne(""), values)
    return out


def coalesce_dates(df: pd.DataFrame, cols: list[str | None]) -> pd.Series:
    out = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    for col in cols:
        if col and col in df.columns:
            parsed = pd.to_datetime(df[col], errors="coerce")
            out = out.fillna(parsed)
    return out


def pct(n: int | float, d: int | float) -> float:
    return round(100 * n / d, 1) if d else 0.0


def trunc(text: Any, limit: int = 500) -> str:
    s = re.sub(r"\\s+", " ", clean(text)).strip()
    return s if len(s) <= limit else s[: limit - 1].rstrip() + "…"


def write_xlsx(path: Path, sheets: dict[str, pd.DataFrame]) -> None:
    path.parent.mkdir(exist_ok=True, parents=True)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for sheet, df in sheets.items():
            df.to_excel(writer, sheet_name=sheet[:31], index=False)
    print("saved:", path)


def column_inventory(df: pd.DataFrame, source: str, max_examples: int = 4) -> pd.DataFrame:
    rows = []
    total = len(df)
    for col in df.columns:
        s = df[col].map(clean)
        filled = s.ne("")
        examples = s[filled].drop_duplicates().head(max_examples).tolist()
        rows.append({
            "Источник": source,
            "Поле": col,
            "Заполнено, N": int(filled.sum()),
            "Заполнено, %": pct(int(filled.sum()), total),
            "Уникальных значений": int(s[filled].nunique()),
            "Примеры значений": " | ".join(map(lambda x: trunc(x, 120), examples)),
            "Предварительное решение": "",
            "Комментарий": "",
        })
    return pd.DataFrame(rows).sort_values(["Заполнено, %", "Уникальных значений"], ascending=[False, False])

## 3. Собираем минимальный Voice-слой

In [ ]:
def read_excel_required(path: Path, label: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Не найден {label}: {path}")
    df = pd.read_excel(path, dtype=str, engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]
    print(label, path.name, df.shape)
    return df


dialogues = read_excel_required(DIALOGUES_XLSX, "диалоги")
days = read_excel_required(DAYS_XLSX, "дни")
match_keys = read_excel_required(MATCH_XLSX, "ключи для матчинга")

# Даты звонка из дни.xlsx.
days_ucid = find_col(days, exact=["ucid", "uc"], regex=[r"^ucid$", r"^uc$"], required=True, label="days ucid")
days_date = find_col(days, exact=["Дата звонка", "День", "date_time", "Дата"], regex=[r"датазвон", r"день", r"datetime", r"^дата$"], required=True, label="days date")
days_small = days[[days_ucid, days_date]].rename(columns={days_ucid: "ucid", days_date: "Дата звонка"}).copy()
days_small["ucid"] = days_small["ucid"].map(norm_id)
days_small["Дата звонка"] = pd.to_datetime(days_small["Дата звонка"], errors="coerce")
days_small = days_small.drop_duplicates("ucid")

# Ключи из Omega: ucid + Id задачи + Id ПрПр.
match_ucid = find_col(match_keys, exact=["ucid", "uc"], regex=[r"^ucid$", r"^uc$"], required=True, label="match ucid")
match_task = find_col(match_keys, exact=["Id задачи", "id_task", "ed задачи"], regex=[r"(id|ed)задач", r"(id|ed)task", r"taskid"], required=True, label="match Id задачи")
match_offer = find_col(match_keys, exact=["Id ПрПр", "id_prpr", "ed ПрПр"], regex=[r"(id|ed)прпр", r"prpr", r"productoffer"], required=True, label="match Id ПрПр")
match_small = match_keys[[match_ucid, match_task, match_offer]].rename(
    columns={match_ucid: "ucid", match_task: "Id задачи", match_offer: "Id ПрПр"}
).copy()
for col in ["ucid", "Id задачи", "Id ПрПр"]:
    match_small[col] = match_small[col].map(norm_id)
match_small = match_small.drop_duplicates("ucid")

# Основной файл диалогов.
dialogs_ucid = find_col(dialogues, exact=["ucid"], regex=[r"^ucid$"], required=True, label="dialogues ucid")
dialogues = dialogues.rename(columns={dialogs_ucid: "ucid"}).copy()
dialogues["ucid"] = dialogues["ucid"].map(norm_id)

voice = dialogues.merge(days_small, on="ucid", how="left", validate="m:1")
voice = voice.merge(match_small, on="ucid", how="left", validate="m:1")
voice = voice[(voice["Дата звонка"] >= CALL_START) & (voice["Дата звонка"] < CALL_END_EXCLUSIVE)].copy()
voice = voice.sort_values(["Дата звонка", "ucid"], na_position="last").reset_index(drop=True)
voice["voice_row"] = voice.index + 2

print("Voice февраль:", voice.shape)
print("Есть Id задачи:", voice["Id задачи"].map(clean).ne("").sum(), f"({pct(voice['Id задачи'].map(clean).ne('').sum(), len(voice))}%)")
print("Есть Id ПрПр:", voice["Id ПрПр"].map(clean).ne("").sum(), f"({pct(voice['Id ПрПр'].map(clean).ne('').sum(), len(voice))}%)")
display(voice.head(3))

## 4. Читаем исходные CRM-файлы

In [ ]:
CRM_READ_KWARGS = {
    "sep": "\\t",
    "encoding": "cp1251",
    "engine": "python",
    "quoting": csv.QUOTE_NONE,
    "dtype": str,
    "keep_default_na": False,
}


def read_crm_csv_with_bad_report(path: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, int | str]]:
    # Read CRM like the earlier Omega notebook and capture true parser-warning skipped rows.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", pd.errors.ParserWarning)
        df = pd.read_csv(
            path,
            **CRM_READ_KWARGS,
            on_bad_lines="warn",
        )
    df.columns = [str(c).strip() for c in df.columns]

    bad_records = []
    for warn in caught:
        msg = str(warn.message)
        if "Skipping line" not in msg:
            continue
        m = re.search(
            r"Skipping line (\\d+): Expected (\\d+) fields in line \\d+, saw (\\d+)",
            msg,
        )
        if not m:
            m = re.search(r"Skipping line (\\d+).*?Expected (\\d+).*?saw (\\d+)", msg)
        if m:
            bad_records.append({
                "Файл": path.name,
                "Номер строки": int(m.group(1)),
                "Ожидалось полей": int(m.group(2)),
                "Фактически полей": int(m.group(3)),
                "Лишних полей": int(m.group(3)) - int(m.group(2)),
                "Сообщение pandas": msg,
            })
        else:
            bad_records.append({
                "Файл": path.name,
                "Номер строки": None,
                "Ожидалось полей": None,
                "Фактически полей": None,
                "Лишних полей": None,
                "Сообщение pandas": msg,
            })

    bad_df = pd.DataFrame(bad_records)
    if not bad_df.empty and bad_df["Номер строки"].notna().any():
        target_lines = set(bad_df["Номер строки"].dropna().astype(int))
        previews = {}
        with path.open("r", encoding="cp1251", errors="replace") as f:
            for line_no, line in enumerate(f, start=1):
                if line_no in target_lines:
                    previews[line_no] = line[:700]
        bad_df["Фрагмент исходной строки"] = bad_df["Номер строки"].map(previews)

    raw_lines = sum(1 for _ in path.open("r", encoding="cp1251", errors="replace"))
    stats = {
        "encoding": "cp1251",
        "separator": "tab",
        "raw_lines": raw_lines,
        "bad_skipped_lines": len(bad_df),
        # Диагностическая физическая разница строк; это не оценка потерь записей.
        "physical_line_delta": max(raw_lines - 1 - len(df), 0),
    }
    return df, bad_df, stats


crm_paths = sorted(BASE_DIR.glob(CRM_GLOB))
if not crm_paths:
    raise FileNotFoundError(f"Не найдены CRM-файлы по маске {CRM_GLOB} в {BASE_DIR}")

crm_frames = {}
crm_read_log = []
crm_bad_lines = []
for path in crm_paths:
    df, bad_df, read_stats = read_crm_csv_with_bad_report(path)
    crm_frames[path.name] = df
    if not bad_df.empty:
        crm_bad_lines.append(bad_df)
    crm_read_log.append({
        "Файл": path.name,
        "Строк в файле, включая заголовок": read_stats.get("raw_lines", 0),
        "Прочитано строк данных": len(df),
        "Колонок": df.shape[1],
        "Реально пропущено строк parser warning": read_stats.get("bad_skipped_lines", 0),
        "Физическая разница строк, не равно потерям": read_stats.get("physical_line_delta", 0),
        "Кодировка": read_stats.get("encoding", ""),
    })
    print(path.name, df.shape)

crm_read_log = pd.DataFrame(crm_read_log)
crm_bad_lines = pd.concat(crm_bad_lines, ignore_index=True) if crm_bad_lines else pd.DataFrame()
display(crm_read_log)
if not crm_bad_lines.empty:
    display(crm_bad_lines)

## 5. Инвентаризация полей

In [ ]:
inventories = [column_inventory(voice, "Voice reconstructed")]
for name, df in crm_frames.items():
    inventories.append(column_inventory(df, f"CRM raw: {name}"))
inventory = pd.concat(inventories, ignore_index=True)
display(inventory.head(40))

write_xlsx(
    OUT_DIR / "01_inventory.xlsx",
    {
        "Инвентаризация": inventory,
        "Лог чтения CRM": crm_read_log,
        "Проблемные строки CRM": crm_bad_lines,
    },
)

## 6. Определяем CRM-файлы задач и продуктовых предложений

In [ ]:
def detect_task_offer_files(frames: dict[str, pd.DataFrame]) -> tuple[str, str]:
    task_candidates = []
    offer_candidates = []
    for name, df in frames.items():
        if find_col(df, exact=["taskid"], regex=[r"^taskid$"]):
            task_candidates.append(name)
        if find_col(df, exact=["productOfferId", "productofferid"], regex=[r"productofferid"]):
            offer_candidates.append(name)
    if not task_candidates:
        raise KeyError("Не найден CRM-файл с taskid.")
    if not offer_candidates:
        raise KeyError("Не найден CRM-файл с productOfferId.")
    return task_candidates[0], offer_candidates[0]


task_file, offer_file = detect_task_offer_files(crm_frames)
tasks = crm_frames[task_file].copy()
offers = crm_frames[offer_file].copy()
tasks["_crm_row_index"] = tasks.index + 2
offers["_crm_row_index"] = offers.index + 2

print("Tasks:", task_file, tasks.shape)
print("Offers:", offer_file, offers.shape)

task_cols = {
    "taskid": find_col(tasks, exact=["taskid"], required=True),
    "key": find_col(tasks, exact=["key"]),
    "ucpid": find_col(tasks, exact=["ucpid"], required=True),
    "contactucpid": find_col(tasks, exact=["contactucpid"]),
    "taskresult": find_col(tasks, exact=["taskresult"]),
    "taskresultcomment": find_col(tasks, exact=["taskresultcomment"]),
    "tasksummary": find_col(tasks, exact=["tasksummary"]),
    "taskdescription": find_col(tasks, exact=["taskdescription"]),
    "taskcomment": find_col(tasks, exact=["taskcomment"]),
    "taskdeactivationcomment": find_col(tasks, exact=["taskdeactivationcomment"]),
    "tasktype": find_col(tasks, exact=["tasktype"]),
    "taskcategory": find_col(tasks, exact=["taskcategory"]),
    "taskchannelcode": find_col(tasks, exact=["taskchannelcode"]),
    "tasksourcecode": find_col(tasks, exact=["tasksourcecode"]),
    "taskscenario": find_col(tasks, exact=["taskscenario"]),
    "taskstatus": find_col(tasks, exact=["taskstatus"]),
    "taskstatecode": find_col(tasks, exact=["taskstatecode"]),
    "statusofcooperation": find_col(tasks, exact=["statusofcooperation"]),
    "productcode": find_col(tasks, exact=["productcode"]),
    "productgroupcode": find_col(tasks, exact=["productgroupcode"]),
    "creationtime": find_col(tasks, exact=["creationtime"]),
    "planneddate": find_col(tasks, exact=["planneddate"]),
    "planstartdate": find_col(tasks, exact=["planstartdate"]),
    "factstartdate": find_col(tasks, exact=["factstartdate"]),
    "factenddate": find_col(tasks, exact=["factenddate"]),
    "updatetime": find_col(tasks, exact=["updatetime"]),
}

offer_cols = {
    "productOfferId": find_col(offers, exact=["productOfferId", "productofferid"], required=True),
    "key": find_col(offers, exact=["key"]),
    "ucpId": find_col(offers, exact=["ucpId", "ucpid"], required=True),
    "corpOfferId": find_col(offers, exact=["corpOfferId"]),
    "dealId": find_col(offers, exact=["dealId"]),
    "leadId": find_col(offers, exact=["leadId"]),
    "parentLeadId": find_col(offers, exact=["parentLeadId"]),
    "productCode": find_col(offers, exact=["productCode"]),
    "productGroupCode": find_col(offers, exact=["productGroupCode"]),
    "productOfferStatusCode": find_col(offers, exact=["productOfferStatusCode"]),
    "offerStateCode": find_col(offers, exact=["offerStateCode"]),
    "sbrfSaleStageCode": find_col(offers, exact=["sbrfSaleStageCode"]),
    "sbrfStageDidCode": find_col(offers, exact=["sbrfStageDidCode"]),
    "sbrfOfferTypeCode": find_col(offers, exact=["sbrfOfferTypeCode"]),
    "sbrfOfferChannelCode": find_col(offers, exact=["sbrfOfferChannelCode"]),
    "sbrfOfferSourceCode": find_col(offers, exact=["sbrfOfferSourceCode"]),
    "sbrfOfferScenarioCode": find_col(offers, exact=["sbrfOfferScenarioCode"]),
    "sbrfOfferRefusalCode": find_col(offers, exact=["sbrfOfferRefusalCode"]),
    "sbrfFormalizationType": find_col(offers, exact=["sbrfFormalizationType"]),
    "productOfferDescription": find_col(offers, exact=["productOfferDescription"]),
    "closingComment": find_col(offers, exact=["closingComment"]),
    "offerDeactivationComment": find_col(offers, exact=["offerDeactivationComment"]),
    "creationTime": find_col(offers, exact=["creationTime"]),
    "offerStartDate": find_col(offers, exact=["offerStartDate"]),
    "offerEndDate": find_col(offers, exact=["offerEndDate"]),
    "offerDeactivationDate": find_col(offers, exact=["offerDeactivationDate"]),
    "updateTime": find_col(offers, exact=["updateTime"]),
}

display(pd.DataFrame({"task_key": list(task_cols), "task_column": list(task_cols.values())}))
display(pd.DataFrame({"offer_key": list(offer_cols), "offer_column": list(offer_cols.values())}))

## 7. Матчим Voice с CRM по Id задачи и Id ПрПр

In [ ]:
tasks["_taskid_norm"] = tasks[task_cols["taskid"]].map(norm_id)
tasks["_ucpid_norm"] = tasks[task_cols["ucpid"]].map(norm_id)
if task_cols.get("key"):
    tasks["_task_key_norm"] = tasks[task_cols["key"]].map(norm_id)
else:
    tasks["_task_key_norm"] = ""

offers["_offerid_norm"] = offers[offer_cols["productOfferId"]].map(norm_id)
offers["_ucpid_norm"] = offers[offer_cols["ucpId"]].map(norm_id)
if offer_cols.get("key"):
    offers["_offer_key_norm"] = offers[offer_cols["key"]].map(norm_id)
else:
    offers["_offer_key_norm"] = ""

bridge = voice[["ucid", "Дата звонка", "Id задачи", "Id ПрПр"]].copy()
bridge["voice_task_id_norm"] = bridge["Id задачи"].map(norm_id)
bridge["voice_offer_id_norm"] = bridge["Id ПрПр"].map(norm_id)

def build_object_lookup(
    df: pd.DataFrame,
    lookup_specs: list[tuple[str, str]],
    ucp_col: str,
    key_col: str,
    row_col: str,
    object_col: str,
    object_label: str,
) -> pd.DataFrame:
    parts = []
    for match_by, lookup_col in lookup_specs:
        if not lookup_col or lookup_col not in df.columns:
            continue
        part = pd.DataFrame({
            "lookup_value": df[lookup_col].map(norm_id),
            "matched_ucpid": df[ucp_col].map(norm_id),
            f"matched_{object_label}_id": df[object_col].map(norm_id),
            f"matched_{object_label}_key": df[key_col].map(norm_id) if key_col in df.columns else "",
            f"matched_{object_label}_row": df[row_col],
            f"{object_label}_match_by": match_by,
        })
        part = part[part["lookup_value"].map(clean).ne("")].copy()
        if not part.empty:
            parts.append(part)
    if not parts:
        return pd.DataFrame(columns=[
            "lookup_value", "matched_ucpid", f"matched_{object_label}_id", f"matched_{object_label}_key",
            f"matched_{object_label}_row", f"{object_label}_match_by", f"{object_label}_match_count",
        ])
    lookup = pd.concat(parts, ignore_index=True)
    counts = lookup.groupby("lookup_value").size().rename(f"{object_label}_match_count").reset_index()
    first = lookup.drop_duplicates("lookup_value", keep="first")
    return first.merge(counts, on="lookup_value", how="left")


task_lookup = build_object_lookup(
    tasks,
    [("taskid", "_taskid_norm"), ("task_key", "_task_key_norm")],
    "_ucpid_norm",
    "_task_key_norm",
    "_crm_row_index",
    "_taskid_norm",
    "task",
)
offer_lookup = build_object_lookup(
    offers,
    [("productOfferId", "_offerid_norm"), ("offer_key", "_offer_key_norm")],
    "_ucpid_norm",
    "_offer_key_norm",
    "_crm_row_index",
    "_offerid_norm",
    "offer",
)

bridge = bridge.merge(
    task_lookup,
    left_on="voice_task_id_norm",
    right_on="lookup_value",
    how="left",
).rename(columns={
    "matched_ucpid": "task_ucpid",
}).drop(columns=["lookup_value"], errors="ignore")

bridge = bridge.merge(
    offer_lookup,
    left_on="voice_offer_id_norm",
    right_on="lookup_value",
    how="left",
    suffixes=("", "_offer_lookup"),
).rename(columns={
    "matched_ucpid": "offer_ucpid",
}).drop(columns=["lookup_value"], errors="ignore")

bridge["final_ucpid"] = bridge["task_ucpid"].where(bridge["task_ucpid"].map(clean).ne(""), bridge["offer_ucpid"])
bridge["match_source"] = np.select(
    [
        bridge["task_ucpid"].map(clean).ne("") & bridge["offer_ucpid"].map(clean).ne(""),
        bridge["task_ucpid"].map(clean).ne(""),
        bridge["offer_ucpid"].map(clean).ne(""),
    ],
    ["task+offer", "task", "offer"],
    default="no_bridge",
)
bridge["task_offer_ucpid_conflict"] = (
    bridge["task_ucpid"].map(clean).ne("")
    & bridge["offer_ucpid"].map(clean).ne("")
    & (bridge["task_ucpid"].map(clean) != bridge["offer_ucpid"].map(clean))
)

match_debug = pd.DataFrame([
    {"Показатель": "Февральских звонков Voice", "N": len(bridge), "% от Voice": 100.0},
    {"Показатель": "Voice: непустой Id задачи", "N": int(bridge["voice_task_id_norm"].map(clean).ne("").sum()), "% от Voice": pct(bridge["voice_task_id_norm"].map(clean).ne("").sum(), len(bridge))},
    {"Показатель": "Task: найдено в CRM по Id задачи или key", "N": int(bridge["task_ucpid"].map(clean).ne("").sum()), "% от Voice": pct(bridge["task_ucpid"].map(clean).ne("").sum(), len(bridge))},
    {"Показатель": "Voice: непустой Id ПрПр", "N": int(bridge["voice_offer_id_norm"].map(clean).ne("").sum()), "% от Voice": pct(bridge["voice_offer_id_norm"].map(clean).ne("").sum(), len(bridge))},
    {"Показатель": "ПрПр: найдено в CRM по productOfferId или key", "N": int(bridge["offer_ucpid"].map(clean).ne("").sum()), "% от Voice": pct(bridge["offer_ucpid"].map(clean).ne("").sum(), len(bridge))},
    {"Показатель": "Есть итоговая CRM-связка final_ucpid", "N": int(bridge["final_ucpid"].map(clean).ne("").sum()), "% от Voice": pct(bridge["final_ucpid"].map(clean).ne("").sum(), len(bridge))},
    {"Показатель": "Конфликт task/offer ucpid", "N": int(bridge["task_offer_ucpid_conflict"].sum()), "% от Voice": pct(int(bridge["task_offer_ucpid_conflict"].sum()), len(bridge))},
])

print("Всего февральских звонков:", len(bridge))
print("Есть CRM-связка:", bridge["final_ucpid"].map(clean).ne("").sum(), f"({pct(bridge['final_ucpid'].map(clean).ne('').sum(), len(bridge))}%)")
print("Конфликт task/offer ucpid:", int(bridge["task_offer_ucpid_conflict"].sum()))
display(match_debug)
display(bridge["match_source"].value_counts(dropna=False).rename_axis("match_source").reset_index(name="N"))
display(bridge["task_match_by"].value_counts(dropna=False).rename_axis("task_match_by").reset_index(name="N"))
display(bridge["offer_match_by"].value_counts(dropna=False).rename_axis("offer_match_by").reset_index(name="N"))
display(bridge.head(10))
display(bridge[bridge["final_ucpid"].map(clean).eq("")].head(20))

## 8. Строим единую CRM-историю по найденным клиентам

In [ ]:
def normalize_task_history(tasks: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=tasks.index)
    out["crm_entity"] = "task"
    out["crm_row_index"] = tasks["_crm_row_index"]
    out["crm_object_id"] = tasks[task_cols["taskid"]].map(norm_id)
    out["crm_key"] = tasks["_task_key_norm"]
    out["client_key"] = tasks["_ucpid_norm"]
    out["event_date"] = coalesce_dates(tasks, [
        task_cols.get("factenddate"), task_cols.get("factstartdate"), task_cols.get("updatetime"),
        task_cols.get("creationtime"), task_cols.get("planneddate"), task_cols.get("planstartdate"),
    ])
    out["product_code"] = coalesce_cols(tasks, [task_cols.get("productcode")])
    out["product_group_code"] = coalesce_cols(tasks, [task_cols.get("productgroupcode")])
    out["campaign"] = coalesce_cols(tasks, [find_col(tasks, exact=["taskcampaign"])])
    out["category_code"] = coalesce_cols(tasks, [task_cols.get("taskcategory")])
    out["type_code"] = coalesce_cols(tasks, [task_cols.get("tasktype")])
    out["channel_code"] = coalesce_cols(tasks, [task_cols.get("taskchannelcode")])
    out["source_code"] = coalesce_cols(tasks, [task_cols.get("tasksourcecode")])
    out["scenario_code"] = coalesce_cols(tasks, [task_cols.get("taskscenario")])
    out["status_code"] = coalesce_cols(tasks, [task_cols.get("taskstatus")])
    out["state_code"] = coalesce_cols(tasks, [task_cols.get("taskstatecode")])
    out["result_code"] = coalesce_cols(tasks, [task_cols.get("taskresult")])
    out["cooperation_status"] = coalesce_cols(tasks, [task_cols.get("statusofcooperation")])
    out["stage_code"] = ""
    out["stage_detail_code"] = ""
    out["refusal_code"] = ""
    out["formalization_type"] = ""
    out["creation_time"] = coalesce_cols(tasks, [task_cols.get("creationtime")])
    out["start_time"] = coalesce_cols(tasks, [task_cols.get("factstartdate"), task_cols.get("planstartdate")])
    out["end_time"] = coalesce_cols(tasks, [task_cols.get("factenddate")])
    out["planned_time"] = coalesce_cols(tasks, [task_cols.get("planneddate")])
    out["update_time"] = coalesce_cols(tasks, [task_cols.get("updatetime")])
    out["taskresultcomment"] = coalesce_cols(tasks, [task_cols.get("taskresultcomment")])
    out["tasksummary"] = coalesce_cols(tasks, [task_cols.get("tasksummary")])
    out["taskdescription"] = coalesce_cols(tasks, [task_cols.get("taskdescription")])
    out["taskcomment"] = coalesce_cols(tasks, [task_cols.get("taskcomment")])
    out["taskdeactivationcomment"] = coalesce_cols(tasks, [task_cols.get("taskdeactivationcomment")])
    out["productOfferDescription"] = ""
    out["closingComment"] = ""
    out["offerDeactivationComment"] = ""
    return out


def normalize_offer_history(offers: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=offers.index)
    out["crm_entity"] = "offer"
    out["crm_row_index"] = offers["_crm_row_index"]
    out["crm_object_id"] = offers[offer_cols["productOfferId"]].map(norm_id)
    out["crm_key"] = offers["_offer_key_norm"]
    out["client_key"] = offers["_ucpid_norm"]
    out["event_date"] = coalesce_dates(offers, [
        offer_cols.get("updateTime"), offer_cols.get("offerEndDate"), offer_cols.get("offerDeactivationDate"),
        offer_cols.get("offerStartDate"), offer_cols.get("creationTime"),
    ])
    out["product_code"] = coalesce_cols(offers, [offer_cols.get("productCode")])
    out["product_group_code"] = coalesce_cols(offers, [offer_cols.get("productGroupCode")])
    out["campaign"] = ""
    out["category_code"] = ""
    out["type_code"] = coalesce_cols(offers, [offer_cols.get("sbrfOfferTypeCode")])
    out["channel_code"] = coalesce_cols(offers, [offer_cols.get("sbrfOfferChannelCode")])
    out["source_code"] = coalesce_cols(offers, [offer_cols.get("sbrfOfferSourceCode")])
    out["scenario_code"] = coalesce_cols(offers, [offer_cols.get("sbrfOfferScenarioCode")])
    out["status_code"] = coalesce_cols(offers, [offer_cols.get("productOfferStatusCode")])
    out["state_code"] = coalesce_cols(offers, [offer_cols.get("offerStateCode")])
    out["result_code"] = ""
    out["cooperation_status"] = ""
    out["stage_code"] = coalesce_cols(offers, [offer_cols.get("sbrfSaleStageCode")])
    out["stage_detail_code"] = coalesce_cols(offers, [offer_cols.get("sbrfStageDidCode")])
    out["refusal_code"] = coalesce_cols(offers, [offer_cols.get("sbrfOfferRefusalCode")])
    out["formalization_type"] = coalesce_cols(offers, [offer_cols.get("sbrfFormalizationType")])
    out["creation_time"] = coalesce_cols(offers, [offer_cols.get("creationTime")])
    out["start_time"] = coalesce_cols(offers, [offer_cols.get("offerStartDate")])
    out["end_time"] = coalesce_cols(offers, [offer_cols.get("offerEndDate"), offer_cols.get("offerDeactivationDate")])
    out["planned_time"] = ""
    out["update_time"] = coalesce_cols(offers, [offer_cols.get("updateTime")])
    out["taskresultcomment"] = ""
    out["tasksummary"] = ""
    out["taskdescription"] = ""
    out["taskcomment"] = ""
    out["taskdeactivationcomment"] = ""
    out["productOfferDescription"] = coalesce_cols(offers, [offer_cols.get("productOfferDescription")])
    out["closingComment"] = coalesce_cols(offers, [offer_cols.get("closingComment")])
    out["offerDeactivationComment"] = coalesce_cols(offers, [offer_cols.get("offerDeactivationComment")])
    return out


task_hist = normalize_task_history(tasks)
offer_hist = normalize_offer_history(offers)
crm_all = pd.concat([task_hist, offer_hist], ignore_index=True)
crm_all = crm_all[
    (crm_all["event_date"].isna())
    | ((crm_all["event_date"] >= CRM_HISTORY_START) & (crm_all["event_date"] < CRM_HISTORY_END_EXCLUSIVE))
].copy()

calls = bridge[["ucid", "Дата звонка", "final_ucpid", "match_source"]].copy()
calls = calls[calls["final_ucpid"].map(clean).ne("")].drop_duplicates("ucid")
crm_history = calls.merge(crm_all, left_on="final_ucpid", right_on="client_key", how="left")
crm_history["history_timing"] = np.select(
    [
        crm_history["event_date"].notna() & crm_history["Дата звонка"].notna() & (crm_history["event_date"] < crm_history["Дата звонка"]),
        crm_history["event_date"].notna() & crm_history["Дата звонка"].notna() & (crm_history["event_date"] >= crm_history["Дата звонка"]),
    ],
    ["before_call", "current_or_after_call"],
    default="unknown_date",
)
crm_history["days_before_call"] = (crm_history["Дата звонка"] - crm_history["event_date"]).dt.total_seconds() / 86400
crm_text_cols = [
    "taskresultcomment", "tasksummary", "taskdescription", "taskcomment", "taskdeactivationcomment",
    "productOfferDescription", "closingComment", "offerDeactivationComment",
]
crm_text_cols = [col for col in crm_text_cols if col in crm_history.columns]
if crm_text_cols:
    crm_history["text_combined"] = crm_history[crm_text_cols].apply(
        lambda row: re.sub(r"\\s+", " ", " ".join(clean(v) for v in row)).strip(),
        axis=1,
    )
else:
    crm_history["text_combined"] = ""
crm_history["is_salary_event"] = (
    crm_history["product_group_code"]
    .map(clean)
    .astype(str)
    .str.contains("SALARY|ЗАРПЛАТ", case=False, na=False)
)
salary_search_cols = ["product_group_code", "product_code", "text_combined", "status_code", "stage_code", "result_code"]
salary_search_cols = [col for col in salary_search_cols if col in crm_history.columns]
if salary_search_cols:
    salary_search_text = crm_history[salary_search_cols].apply(
        lambda row: re.sub(r"\\s+", " ", " ".join(clean(v) for v in row)).strip(),
        axis=1,
    )
else:
    salary_search_text = pd.Series("", index=crm_history.index)
salary_search_text = salary_search_text.map(clean).astype(str)
crm_history["has_salary_text"] = salary_search_text.str.contains(
    r"зарплат|\\bзп\\b|salary|реестр|сотрудник|ведомост|карты сотруд",
    case=False,
    na=False,
)

before = crm_history[crm_history["history_timing"].eq("before_call")].copy()
before = before.sort_values(["ucid", "event_date"], ascending=[True, False])
before["before_rank"] = before.groupby("ucid").cumcount() + 1
crm_history = crm_history.merge(before[["ucid", "crm_entity", "crm_row_index", "before_rank"]], on=["ucid", "crm_entity", "crm_row_index"], how="left")

print("CRM history rows:", len(crm_history))
print("before_call rows:", int(crm_history["history_timing"].eq("before_call").sum()))
print("ucid с before_call:", crm_history.loc[crm_history["history_timing"].eq("before_call"), "ucid"].nunique())
display(crm_history.head(5))

## 9. Базовые статистики и глубина истории

In [ ]:
def build_depth_stats(crm_history: pd.DataFrame) -> pd.DataFrame:
    before = crm_history[crm_history["history_timing"].eq("before_call")].copy()
    base = before["ucid"].nunique()
    rows = []
    windows = [("последний 1 день", 1), ("последние 5 дней", 5), ("последние 30 дней", 30), ("вся история до звонка", None)]
    for label, days in windows:
        part = before if days is None else before[before["days_before_call"].between(0, days, inclusive="both")]
        rows.append({
            "Окно": label,
            "Звонков с любой CRM-историей": part["ucid"].nunique(),
            "% от звонков с CRM-историей": pct(part["ucid"].nunique(), base),
            "CRM-событий": len(part),
            "Звонков с событием SALARY": part.loc[part["is_salary_event"], "ucid"].nunique(),
            "Звонков с SALARY или ЗП-текстом": part.loc[part["is_salary_event"] | part["has_salary_text"], "ucid"].nunique(),
            "Звонков с текстовым комментарием": part.loc[part["text_combined"].map(clean).ne(""), "ucid"].nunique(),
        })
    return pd.DataFrame(rows)


stats = pd.DataFrame([
    {"Показатель": "Февральских звонков Voice", "N": len(voice), "%": 100.0, "База": "voice"},
    {"Показатель": "Есть связка с CRM", "N": bridge["final_ucpid"].map(clean).ne("").sum(), "%": pct(bridge["final_ucpid"].map(clean).ne("").sum(), len(bridge)), "База": "voice"},
    {"Показатель": "Есть CRM-история до звонка", "N": crm_history.loc[crm_history["history_timing"].eq("before_call"), "ucid"].nunique(), "%": pct(crm_history.loc[crm_history["history_timing"].eq("before_call"), "ucid"].nunique(), len(voice)), "База": "voice"},
    {"Показатель": "Есть SALARY-событие до звонка", "N": crm_history.loc[crm_history["history_timing"].eq("before_call") & crm_history["is_salary_event"], "ucid"].nunique(), "%": pct(crm_history.loc[crm_history["history_timing"].eq("before_call") & crm_history["is_salary_event"], "ucid"].nunique(), len(voice)), "База": "voice"},
    {"Показатель": "Есть SALARY или ЗП-текст до звонка", "N": crm_history.loc[crm_history["history_timing"].eq("before_call") & (crm_history["is_salary_event"] | crm_history["has_salary_text"]), "ucid"].nunique(), "%": pct(crm_history.loc[crm_history["history_timing"].eq("before_call") & (crm_history["is_salary_event"] | crm_history["has_salary_text"]), "ucid"].nunique(), len(voice)), "База": "voice"},
])
depth_stats = build_depth_stats(crm_history)

display(stats)
display(depth_stats)

write_xlsx(
    OUT_DIR / "02_matching_and_history_stats.xlsx",
    {
        "Статистика": stats,
        "Глубина истории": depth_stats,
        "Лог матчинга": match_debug,
        "Без CRM-связки": bridge[bridge["final_ucpid"].map(clean).eq("")].head(200),
        "Bridge": bridge,
    },
)

## 10. Сохраняем подготовленные рабочие таблицы

In [ ]:
voice.to_excel(OUT_DIR / "03_voice_reconstructed_february.xlsx", index=False)
crm_history.to_excel(OUT_DIR / "04_crm_history_by_ucid.xlsx", index=False)
bridge.to_excel(OUT_DIR / "05_voice_crm_bridge.xlsx", index=False)
print("Сохранено:", OUT_DIR)

## 11. Готовим простые варианты истории для LLM

In [ ]:
EVENT_FIELDS_FOR_LLM = [
    "crm_entity", "crm_row_index", "event_date", "days_before_call", "product_group_code",
    "status_code", "stage_code", "result_code", "refusal_code", "type_code",
    "taskresultcomment", "tasksummary", "taskdescription", "taskcomment", "taskdeactivationcomment",
    "productOfferDescription", "closingComment", "offerDeactivationComment",
]


def select_history_for_call(ucid: str, mode: str, max_events: int = MAX_EVENTS_PER_CALL) -> pd.DataFrame:
    part = crm_history[(crm_history["ucid"].eq(ucid)) & (crm_history["history_timing"].eq("before_call"))].copy()
    if part.empty:
        return part
    if mode == "salary_only":
        part = part[part["is_salary_event"]].copy()
    elif mode == "salary_or_text":
        part = part[part["is_salary_event"] | part["has_salary_text"]].copy()
    elif mode == "all_recent":
        part["_salary_priority"] = (part["is_salary_event"] | part["has_salary_text"]).astype(int)
        part = part.sort_values(["_salary_priority", "event_date"], ascending=[False, False]).drop(columns=["_salary_priority"], errors="ignore")
        return part.head(max_events)
    else:
        raise ValueError(f"unknown mode: {mode}")
    return part.sort_values("event_date", ascending=False).head(max_events)


def format_event_for_llm(row: pd.Series, event_no: int) -> dict[str, Any]:
    out = {"event_no": event_no}
    for col in EVENT_FIELDS_FOR_LLM:
        value = row.get(col)
        if isinstance(value, pd.Timestamp):
            value = value.strftime("%Y-%m-%d")
        elif col == "days_before_call" and pd.notna(value):
            value = round(float(value), 1)
        out[col] = clean(value)
    return out


def build_llm_input(ucid: str, mode: str) -> dict[str, Any]:
    call = voice.loc[voice["ucid"].eq(ucid)].iloc[0].to_dict()
    events = select_history_for_call(ucid, mode)
    event_payload = [format_event_for_llm(row, i + 1) for i, (_, row) in enumerate(events.iterrows())]
    return {
        "case_id": f"{ucid}__{mode}",
        "ucid": ucid,
        "history_mode": mode,
        "call_date": str(call.get("Дата звонка", ""))[:10],
        "task_type": clean(call.get("Тип задачи")),
        "product": clean(call.get("Продукт")),
        "outcome": clean(call.get("Результат коммуникации")),
        "dialogue_text": trunc(call.get("Текст"), 4000),
        "events": event_payload,
        "n_events": len(event_payload),
    }


# Стратифицированный маленький сэмпл: берем разные типы истории, чтобы быстро увидеть слабые места.
before = crm_history[crm_history["history_timing"].eq("before_call")].copy()
candidates = pd.DataFrame({"ucid": sorted(before["ucid"].dropna().unique())})
candidates["has_salary"] = candidates["ucid"].map(lambda u: len(select_history_for_call(u, "salary_only")) > 0)
candidates["has_salary_or_text"] = candidates["ucid"].map(lambda u: len(select_history_for_call(u, "salary_or_text")) > 0)
candidates["n_all_recent"] = candidates["ucid"].map(lambda u: len(select_history_for_call(u, "all_recent", 1000)))
candidates["has_many_events"] = candidates["n_all_recent"] >= 8

rng = np.random.default_rng(RANDOM_SEED)
sample_parts = []
strata = [
    candidates[candidates["has_salary"]],
    candidates[(~candidates["has_salary"]) & candidates["has_salary_or_text"]],
    candidates[(~candidates["has_salary_or_text"]) & candidates["has_many_events"]],
    candidates,
]
per_stratum = max(5, SAMPLE_SIZE // len(strata))
for part in strata:
    if len(part):
        sample_parts.append(part.sample(min(per_stratum, len(part)), random_state=RANDOM_SEED))
sample_ucids = pd.concat(sample_parts).drop_duplicates("ucid").head(SAMPLE_SIZE)["ucid"].tolist()

llm_inputs = []
for ucid in sample_ucids:
    for mode in TEST_HISTORY_MODES:
        llm_inputs.append(build_llm_input(ucid, mode))

llm_inputs_df = pd.DataFrame([
    {
        "case_id": x["case_id"],
        "ucid": x["ucid"],
        "history_mode": x["history_mode"],
        "call_date": x["call_date"],
        "n_events": x["n_events"],
        "outcome": x["outcome"],
        "events_preview": json.dumps(x["events"][:3], ensure_ascii=False),
        "dialogue_preview": trunc(x["dialogue_text"], 600),
    }
    for x in llm_inputs
])
display(llm_inputs_df.head(10))
llm_inputs_df.to_excel(OUT_DIR / "06_llm_inputs_sample.xlsx", index=False)
print("LLM input rows:", len(llm_inputs_df))

## 12. Промпт LLM-разметки

In [ ]:
SYSTEM_PROMPT = """
Ты аналитик CRM-истории и диалогов клиентских менеджеров.

Тебе дают:
1. текст реального звонка по зарплатному проекту;
2. несколько CRM-событий, которые были ДО звонка;
3. режим отбора истории.

Нужно оценить две вещи:
А. Что можно было безопасно использовать в начале звонка боту.
Б. Использовал ли живой менеджер в речи какой-либо факт из CRM-истории.

Важно:
- Не придумывай факты, которых нет в CRM-событиях.
- Не считай технический статус полезным фактом, если он ничего не говорит клиенту.
- Если CRM-событие относится к другому продукту, не делай вид, что это факт по зарплатному проекту.
- Если менеджер просто продает по обычному скрипту, это не использование CRM.
- Если менеджер говорит "мы ранее общались", "договаривались", "вы просили перезвонить", "у вас уже начато оформление" — это может быть использованием истории.
- Стоп-сигналы: свежий отказ, явное отсутствие потребности, сервисная проблема, небезопасный или противоречивый контекст.

Верни строго JSON без markdown.
"""

USER_TEMPLATE = """
Разметь один звонок.

Метаданные звонка:
{call_meta}

CRM-события до звонка:
{events_json}

Текст звонка:
{dialogue_text}

Верни JSON с полями:
{{
  "case_id": "...",
  "history_mode": "...",
  "has_useful_crm_context": true/false,
  "recommended_opening_policy": "normal|acknowledge_history|soft_followup|do_not_sell|needs_review",
  "safe_fact_summary": "коротко, что можно использовать; пусто если нельзя",
  "safe_fact_source_event_no": число или null,
  "why_not_use": "если использовать нельзя или не нужно",
  "manager_used_crm_history": "yes|no|unclear",
  "manager_phrase": "фраза менеджера, если есть",
  "prompt_would_help": "yes|no|unclear",
  "field_value": "какой тип поля/события полезен: status/stage/result/refusal/comment/callback/other/none",
  "needs_human_review": true/false,
  "confidence": 0.0-1.0,
  "reason": "1-3 предложения"
}}
"""


def build_prompt_payload(item: dict[str, Any]) -> tuple[str, str]:
    meta = {
        "case_id": item["case_id"],
        "ucid": item["ucid"],
        "history_mode": item["history_mode"],
        "call_date": item["call_date"],
        "task_type": item["task_type"],
        "product": item["product"],
        "outcome": item["outcome"],
        "n_events": item["n_events"],
    }
    user = USER_TEMPLATE.format(
        call_meta=json.dumps(meta, ensure_ascii=False, indent=2),
        events_json=json.dumps(item["events"], ensure_ascii=False, indent=2),
        dialogue_text=item["dialogue_text"],
    )
    return SYSTEM_PROMPT.strip(), user.strip()

## 13. GigaChat API: smoke/benchmark по моделям

In [ ]:
def extract_json_object(text: str) -> dict[str, Any]:
    text = clean(text)
    if not text:
        raise ValueError("empty response")
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\\{.*\\}", text, flags=re.S)
    if not match:
        raise ValueError(f"no JSON object in response: {text[:300]}")
    return json.loads(match.group(0))


def call_gigachat_one(item: dict[str, Any], model: str) -> dict[str, Any]:
    from gigachat import GigaChat
    from gigachat.models import Chat, Messages, MessagesRole

    system, user = build_prompt_payload(item)
    giga = GigaChat(
        cert_file=str(CERT_FILE),
        key_file=str(KEY_FILE),
        base_url=GIGACHAT_BASE_URL,
        model=model,
        verify_ssl_certs=False,
    )
    payload = Chat(
        model=model,
        messages=[
            Messages(role=MessagesRole.SYSTEM, content=system),
            Messages(role=MessagesRole.USER, content=user),
        ],
        temperature=0,
        top_p=0.9,
        max_tokens=1400,
        timeout=LLM_TIMEOUT,
        repetition_penalty=1.05,
    )
    start = time.time()
    response = giga.chat(payload)
    seconds = round(time.time() - start, 2)
    content = response.choices[0].message.content
    usage = getattr(response, "usage", None)
    parsed = extract_json_object(content)
    parsed.update({
        "_model": model,
        "_seconds": seconds,
        "_raw_response": content,
        "_prompt_tokens": getattr(usage, "prompt_tokens", None) if usage else None,
        "_completion_tokens": getattr(usage, "completion_tokens", None) if usage else None,
        "_total_tokens": getattr(usage, "total_tokens", None) if usage else None,
    })
    return parsed


def run_gigachat_batch(items: list[dict[str, Any]], models: list[str]) -> pd.DataFrame:
    out_path = OUT_DIR / "07_gigachat_annotation_results.csv"
    rows = []
    jobs = [(item, model) for model in models for item in items]
    print("jobs:", len(jobs), "workers:", N_WORKERS)
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(call_gigachat_one, item, model): (item, model) for item, model in jobs}
        for i, fut in enumerate(as_completed(futures), start=1):
            item, model = futures[fut]
            try:
                row = fut.result()
                row["_error"] = ""
            except Exception as exc:
                row = {
                    "case_id": item["case_id"],
                    "history_mode": item["history_mode"],
                    "_model": model,
                    "_error": repr(exc),
                }
            rows.append(row)
            pd.DataFrame(rows).to_csv(out_path, index=False, encoding="utf-8-sig")
            if i % 5 == 0 or i == len(jobs):
                print(f"saved {i}/{len(jobs)} -> {out_path.name}")
    return pd.DataFrame(rows)


if RUN_GIGACHAT:
    if not CERT_FILE.exists() or not KEY_FILE.exists():
        raise FileNotFoundError(f"Не найдены сертификаты: {CERT_FILE} / {KEY_FILE}")
    results = run_gigachat_batch(llm_inputs, GIGACHAT_MODELS)
    display(results.head())
else:
    print("RUN_GIGACHAT=False: API-прогон пропущен. Проверьте входы, затем включите флаг в настройках.")

## 14. Анализ результатов LLM-прогона

In [ ]:
results_path = OUT_DIR / "07_gigachat_annotation_results.csv"
if results_path.exists():
    results = pd.read_csv(results_path, dtype=str, encoding="utf-8-sig")
    print("results:", results.shape)
    display(results.head())

    # Нормализуем несколько булевых полей для сводки.
    for col in ["has_useful_crm_context", "needs_human_review"]:
        if col in results.columns:
            results[col + "_bool"] = results[col].astype(str).str.lower().isin(["true", "1", "yes", "да"])

    summary_rows = []
    for model, part in results.groupby("_model", dropna=False):
        summary_rows.append({
            "model": model,
            "N": len(part),
            "errors": int(part.get("_error", pd.Series([""] * len(part))).map(clean).ne("").sum()),
            "median_seconds": pd.to_numeric(part.get("_seconds"), errors="coerce").median(),
            "has_useful_context_%": pct(int(part.get("has_useful_crm_context_bool", pd.Series(False, index=part.index)).sum()), len(part)),
            "manager_used_yes_%": pct(int(part.get("manager_used_crm_history", pd.Series("", index=part.index)).astype(str).str.lower().eq("yes").sum()), len(part)),
            "prompt_would_help_yes_%": pct(int(part.get("prompt_would_help", pd.Series("", index=part.index)).astype(str).str.lower().eq("yes").sum()), len(part)),
            "needs_review_%": pct(int(part.get("needs_human_review_bool", pd.Series(False, index=part.index)).sum()), len(part)),
        })
    benchmark = pd.DataFrame(summary_rows)
    display(benchmark)

    write_xlsx(
        OUT_DIR / "08_gigachat_model_benchmark.xlsx",
        {
            "Сводка по моделям": benchmark,
            "Разметка": results,
        },
    )
else:
    print("Пока нет результатов LLM. Запустите блок 13 с RUN_GIGACHAT=True.")

## 15. Минимальный экспорт наружу для сверки с другими LLM

In [ ]:
# Этот файл намеренно не содержит сырые CRM-комментарии и полный текст диалога.
# Его можно использовать как внешний слой для сравнения моделей/методик.
if (OUT_DIR / "07_gigachat_annotation_results.csv").exists():
    results = pd.read_csv(OUT_DIR / "07_gigachat_annotation_results.csv", dtype=str, encoding="utf-8-sig")
    keep = [
        "case_id", "history_mode", "_model",
        "has_useful_crm_context", "recommended_opening_policy", "safe_fact_summary",
        "safe_fact_source_event_no", "why_not_use", "manager_used_crm_history",
        "manager_phrase", "prompt_would_help", "field_value", "needs_human_review",
        "confidence", "reason", "_seconds", "_total_tokens", "_error",
    ]
    external = results[[c for c in keep if c in results.columns]].copy()
    external.to_excel(OUT_DIR / "09_external_minimal_review.xlsx", index=False)
    display(external.head())
    print("saved:", OUT_DIR / "09_external_minimal_review.xlsx")
else:
    print("Нет LLM-результатов для внешнего экспорта.")

## 16. Как интерпретировать результат

Сравниваем не одну “магическую” модель, а режимы:

- `salary_only`: самый простой и защищаемый подход, но он может потерять общие задачи/перезвоны без ЗП-тега;
- `salary_or_text`: компромисс — добавляет события, где ЗП виден в тексте;
- `all_recent`: даёт модели больше контекста, но повышает риск шума и ложной привязки.

Если простой режим даёт качество близкое к широкому режиму, production-логику можно упрощать.
Если широкий режим заметно лучше, значит старый более сложный пайплайн был не “перемудриванием”, а способом не потерять полезные факты.